# 05 · Sentence reranker and field context (Member 4: chatbot)

Phase 6's end-to-end check showed that the chatbot often finds the right passage but copies the wrong sentences.
This notebook trains **our own sentence reranker**: a small multilingual cross-encoder
(`cross-encoder/mmarco-mMiniLMv2-L12-H384-v1`, multilingual MiniLM) fine-tuned to read a question and one sentence
(or table row) together and score whether that sentence answers it.

**Labels are auto-generated, not human-checked.** From the Phase 5 question–passage pairs (train split only), a
sentence of the expected passage is *relevant* if it contains a key fact of the expected answer (a number or variety
code that is in both the expected answer and the expected passage); other sentences of that passage, and sentences
of the other top-ranked (look-alike) passages, are *not relevant*. The questions themselves are AI-drafted,
template-generated or machine-translated (DATASET.md).

Everything is chosen on the **validation split**: the epoch, the selection settings, the age-class bonus and the
field-profile boost. The AI-drafted test split is reported once at the end. **The 60-question evaluation set is not
used here.** Uses the project code (`member4_chatbot`), like notebook 03.

In [1]:
RETRAIN = False
SEED = 20261018
BASE = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"
EPOCHS, BATCH, LR, WARMUP, MAX_LEN = 2, 32, 2e-5, 0.1, 256
NEG_PASSAGES, NEG_UNITS_PER_PASSAGE = 3, 8

In [2]:
import sys, json, random, time, itertools, shutil, tempfile
from pathlib import Path
HERE = Path.cwd(); DATA = HERE / "data"
sys.path.insert(0, str(HERE.parent))
OUT = HERE.parent / "artifacts" / "reranker"
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from datasets import Dataset
from sentence_transformers.cross_encoder import CrossEncoder, CrossEncoderTrainer, CrossEncoderTrainingArguments
from sentence_transformers.cross_encoder.losses import BinaryCrossEntropyLoss
from transformers import TrainerCallback, set_seed
from member4_chatbot import models
from member4_chatbot.chat.extractive import extract, split_units
from member4_chatbot.chat.rerank import key_facts, contains_fact, unit_inputs, choose
from member4_chatbot.chat.retrieval import PassageRanker
from member4_chatbot.chat.profile import question_profile
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); set_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ANS = json.loads((HERE.parent / "artifacts" / "answer" / "config.json").read_text())
print("device:", DEVICE, "| answer config:", ANS)

device: cpu | answer config: {'retrieval_threshold': 0.32, 'max_sentences': 2, 'margin': 0.12, 'same_language_delta': 0.05, 'top_k': 5, 'retrieval_mode': {'en': 'rrf', 'si': 'dense'}, 'chosen_on': 'validation split', 'seed': 20261018}


In [3]:
passages = pd.read_csv(DATA / "passages.csv", keep_default_na=False)
qa = pd.read_csv(DATA / "qa_pairs.csv", keep_default_na=False)
P = passages.set_index("id")
qa["gold"] = [[r.passage_id, *filter(None, r.also_correct.split(";"))] for r in qa.itertuples()]
qa["facts"] = [sorted(key_facts(r.answer, [P.loc[p, "text"] for p in r.gold if p in P.index])) for r in qa.itertuples()]
_, collection = models._retriever()
ranker = PassageRanker.from_chroma(passages, collection)
Q = {r.id: e for r, e in zip(qa.itertuples(), models.embed(qa.question.tolist()))}
print(qa.assign(has_facts=qa.facts.str.len() > 0).pivot_table(index="split", columns="has_facts", values="id", aggfunc="count"))

The tokenizer you are loading from 'E:\soilmind\backend\member4-chatbot\artifacts\retriever\model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


has_facts  False  True 
split                  
test          56    172
train        245    642
val           62    189


## 1. Training pairs (train split only)

In [4]:
def ranked(r, k, boost=0):
    prof = question_profile(r.question)
    return ranker.rank_profiled(Q[r.id], r.question, ANS["retrieval_mode"][r.lang], k, prof, boost)

rows = []
rng = random.Random(SEED)
for r in qa[(qa.split == "train") & (qa.facts.str.len() > 0)].itertuples():
    gold = [p for p in r.gold if p in P.index]
    for pid in gold:
        units = split_units(P.loc[pid, "text"], P.loc[pid, "title"])
        labels = [any(contains_fact(u.text, f) for f in r.facts) for u in units]
        if not any(labels):
            continue
        for x, y in zip(unit_inputs(units, P.loc[pid, "title"]), labels):
            rows.append((r.question, x, float(y), "gold"))
    negs = [h.passage_id for h in ranked(r, NEG_PASSAGES + len(gold)) if h.passage_id not in gold][:NEG_PASSAGES]
    for pid in negs:
        units = split_units(P.loc[pid, "text"], P.loc[pid, "title"])
        inputs = unit_inputs(units, P.loc[pid, "title"])
        for x in rng.sample(inputs, min(NEG_UNITS_PER_PASSAGE, len(inputs))):
            rows.append((r.question, x, 0.0, "hard_negative"))
pairs = pd.DataFrame(rows, columns=["question", "sentence", "label", "kind"]).drop_duplicates(["question", "sentence"])
print(f"{len(pairs)} pairs, {int(pairs.label.sum())} relevant ({pairs.label.mean():.1%})")
pairs.groupby("kind").label.agg(["count", "mean"])

24523 pairs, 2934 relevant (12.0%)


,count,mean
kind,,
gold,11217,0.261567
hard_negative,13306,0.000000


## 2. How answers are scored (validation)
For every validation question with key facts: retrieve, pick sentences, and check the shown answer.
**All facts** = every key fact appears (the end-to-end "answer correct" criterion); **fact recall** = share of key
facts shown; **words** = answer length.

In [5]:
VAL = qa[(qa.split == "val") & (qa.facts.str.len() > 0)].reset_index(drop=True)
TEST = qa[(qa.split == "test") & (qa.facts.str.len() > 0)].reset_index(drop=True)

def score_answer(r, pid, lines):
    text = "\n".join(lines)
    found = [contains_fact(text, f) for f in r.facts]
    return {"lang": r.lang, "passage_ok": pid in r.gold, "fact_recall": np.mean(found), "all_facts": all(found),
            "words": len(text.split())}

def run_current(frame, boost=0):
    out = []
    for r in frame.itertuples():
        hits = ranked(r, 5, boost)
        lead = hits[0].score
        h = next((x for x in hits if x.score >= lead - ANS["same_language_delta"] and x.lang == r.lang), hits[0])
        ex = extract(h.passage_id, P.loc[h.passage_id, "text"], P.loc[h.passage_id, "title"], Q[r.id], models.embed,
                     max_sentences=ANS["max_sentences"], margin=ANS["margin"])
        out.append(score_answer(r, h.passage_id, ex.lines))
    return pd.DataFrame(out)

def make_scorer(model):
    cache = {}
    def scorer(q, texts):
        need = [t for t in dict.fromkeys(texts) if (q, t) not in cache]
        if need:
            for t, s in zip(need, model.predict([(q, t) for t in need], batch_size=64, show_progress_bar=False)):
                cache[(q, t)] = float(s)
        return [cache[(q, t)] for t in texts]
    return scorer

def run_rerank(frame, scorer, n_passages=3, max_units=3, margin=0.2, lang_delta=0.1, age_bonus=0.0, boost=0):
    out = []
    for r in frame.itertuples():
        hits = ranked(r, n_passages, boost)
        ps = [{"id": h.passage_id, "text": P.loc[h.passage_id, "text"], "title": P.loc[h.passage_id, "title"], "lang": h.lang} for h in hits]
        c = choose(r.question, ps, scorer, answer_lang=r.lang, max_units=max_units, margin=margin, lang_delta=lang_delta,
                   age=question_profile(r.question).age, age_bonus=age_bonus)
        out.append(score_answer(r, c.passage_id, c.lines))
    return pd.DataFrame(out)

def summary(S):
    g = {"all": S, "en": S[S.lang == "en"], "si": S[S.lang == "si"]}
    return {k: {"n": len(v), "all_facts": round(v.all_facts.mean(), 4), "fact_recall": round(v.fact_recall.mean(), 4),
                "passage_ok": round(v.passage_ok.mean(), 4), "words": round(v.words.mean(), 1)} for k, v in g.items()}

RESULTS = {"val": {}, "test": {}}
RESULTS["val"]["current_selection"] = summary(run_current(VAL))
pretrained = CrossEncoder(BASE, device=DEVICE, max_length=MAX_LEN, activation_fn=torch.nn.Sigmoid())
RESULTS["val"]["pretrained_cross_encoder"] = summary(run_rerank(VAL, make_scorer(pretrained)))
pd.DataFrame({k: v["all"] for k, v in RESULTS["val"].items()}).T

,n,all_facts,fact_recall,passage_ok,words
current_selection,189.0,0.6190,0.7248,0.6455,103.4
pretrained_cross_encoder,189.0,0.6085,0.7281,0.7619,37.3


## 3. Fine-tune the cross-encoder
Binary cross-entropy (positives up-weighted by the class ratio), linear warm-up, AdamW. After each epoch the
validation *all facts* rate (default selection settings) is measured; the best epoch is kept.

In [6]:
class KeepBest(TrainerCallback):
    def __init__(self, model): self.model, self.best, self.state, self.history, self.best_epoch = model, -1, None, [], 0
    def on_epoch_end(self, args, state, control, **kw):
        s = summary(run_rerank(VAL, make_scorer(self.model)))["all"]
        self.history.append({"epoch": round(state.epoch), **s}); print(self.history[-1])
        OUT.mkdir(parents=True, exist_ok=True)
        with open(OUT / "progress.log", "a", encoding="utf-8") as f:   # visible while the notebook runs
            f.write(f"{time.strftime('%H:%M:%S')} epoch {round(state.epoch)}: {s}\n")
        if s["all_facts"] > self.best:
            self.best, self.best_epoch = s["all_facts"], round(state.epoch)
            self.state = {k: v.detach().cpu().clone() for k, v in self.model.model.state_dict().items()}

if RETRAIN:
    model = CrossEncoder(BASE, device=DEVICE, max_length=MAX_LEN, activation_fn=torch.nn.Sigmoid())
    ds = Dataset.from_dict({"question": pairs.question.tolist(), "sentence": pairs.sentence.tolist(), "label": pairs.label.tolist()})
    pos_weight = torch.tensor((1 - pairs.label.mean()) / pairs.label.mean())
    keep = KeepBest(model)
    tmp = tempfile.mkdtemp()
    args = CrossEncoderTrainingArguments(output_dir=tmp, num_train_epochs=EPOCHS, per_device_train_batch_size=BATCH,
                                         learning_rate=LR, warmup_ratio=WARMUP, seed=SEED, data_seed=SEED,
                                         save_strategy="no", eval_strategy="no", logging_steps=50, report_to="none", disable_tqdm=True)
    trainer = CrossEncoderTrainer(model=model, args=args, train_dataset=ds,
                                  loss=BinaryCrossEntropyLoss(model, pos_weight=pos_weight), callbacks=[keep])
    t0 = time.time(); trainer.train(); TRAIN_SECONDS = time.time() - t0
    shutil.rmtree(tmp, ignore_errors=True)
    model.model.load_state_dict(keep.state)
    HISTORY, BEST_EPOCH = keep.history, keep.best_epoch
    model.save(str(OUT / "model"))   # saved at once, so a later cell cannot lose the trained model
    with open(OUT / "progress.log", "a", encoding="utf-8") as f:
        f.write(f"{time.strftime('%H:%M:%S')} saved best epoch {BEST_EPOCH} ({TRAIN_SECONDS / 60:.1f} min, {len(pairs)} pairs)\n")
    print(f"best epoch {BEST_EPOCH}; training took {TRAIN_SECONDS / 60:.1f} min on {DEVICE}")
else:
    model = CrossEncoder(str(OUT / "model"), device=DEVICE, max_length=MAX_LEN, activation_fn=torch.nn.Sigmoid())
    saved = json.loads((OUT / "metrics.json").read_text())
    HISTORY, BEST_EPOCH, TRAIN_SECONDS = saved["history"], saved["best_epoch"], saved["train_seconds"]
scorer = make_scorer(model)

The tokenizer you are loading from 'E:\soilmind\backend\member4-chatbot\artifacts\reranker\model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


## 4. Selection settings, age-class bonus and field-profile boost (validation)

In [7]:
grid = []
for n_p, mu, mg in itertools.product((1, 3), (2, 3), (0.1, 0.2, 0.4)):
    s = summary(run_rerank(VAL, scorer, n_passages=n_p, max_units=mu, margin=mg))["all"]
    grid.append({"n_passages": n_p, "max_units": mu, "margin": mg, **s})
G = pd.DataFrame(grid).sort_values(["all_facts", "fact_recall", "words"], ascending=[False, False, True])
BEST = G.iloc[0]
SEL = {"n_passages": int(BEST.n_passages), "max_units": int(BEST.max_units), "margin": float(BEST.margin)}
print("chosen:", SEL); G.head(6)

chosen: {'n_passages': 1, 'max_units': 3, 'margin': 0.4}


,n_passages,max_units,margin,n,all_facts,fact_recall,passage_ok,words
5,1,3,0.4,189,0.5397,0.7021,0.6138,43.7
4,1,3,0.2,189,0.5344,0.6970,0.6138,41.5
2,1,2,0.4,189,0.5291,0.6833,0.6138,35.3
11,3,3,0.4,189,0.5238,0.7047,0.5132,43.0
1,1,2,0.2,189,0.5238,0.6807,0.6138,34.5
3,1,3,0.1,189,0.5238,0.6796,0.6138,37.7


**Age-class bonus and field-profile boost.** Validation fertilizer questions name their zone, water regime and
age class, so the *question's* profile is used here as a stand-in for the plot's (in the chat, the plot's profile
fills whatever the question leaves out). Both are chosen on validation fertilizer questions.

In [8]:
FERT_VAL = VAL[VAL.intent.isin(["fertilizer_amount", "fertilizer_timing"])].reset_index(drop=True)
tune = []
for boost, ab in itertools.product((0, 1, 3, 5), (0.0, 0.1, 0.3)):
    s = summary(run_rerank(FERT_VAL, scorer, **SEL, age_bonus=ab, boost=boost))["all"]
    tune.append({"profile_boost": boost, "age_bonus": ab, **s})
TU = pd.DataFrame(tune).sort_values(["all_facts", "passage_ok", "profile_boost", "age_bonus"], ascending=[False, False, True, True])
PROFILE_BOOST, AGE_BONUS = int(TU.iloc[0].profile_boost), float(TU.iloc[0].age_bonus)
print(f"{len(FERT_VAL)} validation fertilizer questions; chosen boost = {PROFILE_BOOST} rank positions, age bonus = {AGE_BONUS}")
TU.head(6)

100 validation fertilizer questions; chosen boost = 3 rank positions, age bonus = 0.3


,profile_boost,age_bonus,n,all_facts,fact_recall,passage_ok,words
8,3,0.3,100,0.67,0.8740,0.91,41.9
11,5,0.3,100,0.67,0.8740,0.91,41.9
5,1,0.3,100,0.61,0.8423,0.71,42.1
6,3,0.0,100,0.60,0.8428,0.91,60.7
7,3,0.1,100,0.60,0.8534,0.91,56.2
9,5,0.0,100,0.60,0.8428,0.91,60.7


## 5. Fair comparison of the three sentence pickers (validation) → choose one
The comparison in section 2 was not like for like: the current selection shows whole tables (about 100 words) and the
pretrained cross-encoder used default settings. Here all three pickers use **the same display** (at most
`max_units` sentences / rows plus their table header and group row), the same settings, the age bonus, and the
field-profile boost for fertilizer questions:
- `retriever_cosine`: our fine-tuned retriever's cosine similarity per sentence (what the chat used before)
- `pretrained_cross_encoder`: the open mMiniLM cross-encoder, not fine-tuned
- `fine_tuned_reranker`: ours

The picker with the best validation *all facts* is used by the chat (ties: ours, then the shorter answers).
The settings grid above was tuned with our fine-tuned reranker, which favours it slightly.

In [9]:
_emb_cache = {}
def cosine_scorer(q, texts):
    need = [t for t in dict.fromkeys([q] + list(texts)) if t not in _emb_cache]
    if need:
        for t, e in zip(need, models.embed(need)): _emb_cache[t] = e
    qe = _emb_cache[q]
    return [float(_emb_cache[t] @ qe) for t in texts]

FERT = ("fertilizer_amount", "fertilizer_timing")
def run_picker(frame, sc):
    parts = [run_rerank(frame[frame.intent.isin(FERT)], sc, **SEL, age_bonus=AGE_BONUS, boost=PROFILE_BOOST),
             run_rerank(frame[~frame.intent.isin(FERT)], sc, **SEL, age_bonus=AGE_BONUS, boost=0)]
    return pd.concat(parts, ignore_index=True)

PICKERS = {"retriever_cosine": cosine_scorer, "pretrained_cross_encoder": make_scorer(pretrained), "fine_tuned_reranker": scorer}
FAIR = {"val": {name: summary(run_picker(VAL, sc)) for name, sc in PICKERS.items()}}
F = pd.DataFrame({k: v["all"] for k, v in FAIR["val"].items()}).T
order = sorted(PICKERS, key=lambda k: (-FAIR["val"][k]["all"]["all_facts"], k != "fine_tuned_reranker", FAIR["val"][k]["all"]["words"]))
PICKER = order[0]
print("chosen picker (validation):", PICKER)
F

chosen picker (validation): fine_tuned_reranker


,n,all_facts,fact_recall,passage_ok,words
retriever_cosine,189.0,0.5873,0.7226,0.8095,38.2
pretrained_cross_encoder,189.0,0.6349,0.7550,0.8095,37.5
fine_tuned_reranker,189.0,0.6455,0.7552,0.8095,35.7


## 6. Test split (AI-drafted questions, used once)

In [10]:
FAIR["test"] = {name: summary(run_picker(TEST, sc)) for name, sc in PICKERS.items()}
FAIR["test"]["current_selection_whole_tables (before)"] = summary(run_current(TEST))
FAIR["val"]["current_selection_whole_tables (before)"] = RESULTS["val"]["current_selection"]
rows = [{"split": sp, "picker": m, "lang": l, **v} for sp in ("val", "test") for m, d in FAIR[sp].items() for l, v in d.items()]
T = pd.DataFrame(rows)
T.pivot_table(index=["split", "picker"], columns="lang", values=["all_facts", "words"]).round(3)

all_facts                words  \
lang                                                all     en     si    all   
split picker                                                                   
test  current_selection_whole_tables (before)     0.552  0.536  0.568  100.3   
      fine_tuned_reranker                         0.593  0.655  0.534   37.7   
      pretrained_cross_encoder                    0.570  0.619  0.523   40.8   
      retriever_cosine                            0.471  0.524  0.420   41.2   
val   current_selection_whole_tables (before)     0.619  0.571  0.670  103.4   
      fine_tuned_reranker                         0.646  0.674  0.615   35.7   
      pretrained_cross_encoder                    0.635  0.643  0.626   37.5   
      retriever_cosine                            0.587  0.622  0.550   38.2   

                                                            
lang                                             en     si  
split picker                                                
test  current_selection_whole_tables (before)  91.8  108.5  
      fine_tuned_reranker                      36.6   38.8  
      pretrained_cross_encoder                 38.2   43.2  
      retriever_cosine                         41.2   41.2  
val   current_selection_whole_tables (before)  90.6  117.2  
      fine_tuned_reranker                      33.0   38.6  
      pretrained_cross_encoder                 34.4   41.0  
      retriever_cosine                         39.0   37.3

In [11]:
t0 = time.perf_counter()
r0 = TEST.iloc[0]
chosen_scorer = PICKERS[PICKER]
for _ in range(20):
    hits = ranker.rank(Q[r0.id], r0.question, ANS["retrieval_mode"][r0.lang], SEL["n_passages"])
    ps = [{"id": h.passage_id, "text": P.loc[h.passage_id, "text"], "title": P.loc[h.passage_id, "title"], "lang": h.lang} for h in hits]
    choose(r0.question + str(_), ps, chosen_scorer, answer_lang=r0.lang, max_units=SEL["max_units"], margin=SEL["margin"], lang_delta=0.1)
RERANK_MS = (time.perf_counter() - t0) / 20 * 1000
print(f"choosing sentences for one question on {DEVICE} ({PICKER}): {RERANK_MS:.0f} ms")

choosing sentences for one question on cpu (fine_tuned_reranker): 833 ms


## 7. Save (the chat loads `config.json`)

In [12]:
config = {**SEL, "picker": PICKER, "lang_delta": 0.1, "age_bonus": AGE_BONUS, "profile_boost": PROFILE_BOOST,
          "max_length": MAX_LEN, "base_model": BASE, "chosen_on": "validation split", "seed": SEED}
metrics = {"config": config, "results": RESULTS, "fair_comparison": FAIR, "validation_grid": grid,
           "validation_profile_tuning": tune, "history": HISTORY, "best_epoch": BEST_EPOCH, "train_seconds": round(TRAIN_SECONDS),
           "training_pairs": int(len(pairs)), "rerank_ms_cpu": round(RERANK_MS),
           "data_warning": "trained and evaluated on auto-generated labels over AI-drafted / template / machine-translated questions; not human-checked"}
OUT.mkdir(parents=True, exist_ok=True)
(OUT / "config.json").write_text(json.dumps(config, indent=2))
(OUT / "metrics.json").write_text(json.dumps(metrics, indent=2))
if True:
    tv, tt = FAIR["val"], FAIR["test"]
    def cell(d, m, l): return f'{d[m][l]["all_facts"]} / {d[m][l]["fact_recall"]} / {d[m][l]["words"]}'
    card = f'''# Model card: SoilMind sentence reranker

**What it does.** After the retriever has found the top passages, scores every sentence / table row of them against
the question and picks the 1-{SEL["max_units"]} that answer it (the chat then shows them with their table header and a
"Show full passage" link). Also nudges fertilizer questions towards documents matching the plot's zone, water regime
and age class (from Member 2's API; the question's own words win).

**Model.** `{BASE}` (multilingual MiniLM cross-encoder) fine-tuned with binary cross-entropy, {EPOCHS} epochs max
(best epoch {BEST_EPOCH}, by validation), lr {LR}, max {MAX_LEN} tokens, seed {SEED}. Trained in {TRAIN_SECONDS / 60:.1f} min on {DEVICE}.

**Training data is auto-generated and NOT human-checked.** {len(pairs)} (question, sentence) pairs from the Phase 5
train split. A sentence is labelled relevant if it contains a key fact (number or variety code) of the expected
answer; sentences of look-alike passages are negatives. The questions themselves are AI-drafted, template-generated
or machine-translated (DATASET.md). Settings were chosen on the validation split; the 60-question evaluation set was
not used.

**Results** (share of answers showing *all* key facts / mean key-fact recall / mean words), all pickers with the
same short display and settings:

| Picker | Validation EN | Validation SI | Test EN | Test SI |
|---|---|---|---|---|
''' + "\n".join(f"| {m}{' **(used)**' if m == PICKER else ''} | {cell(tv, m, 'en')} | {cell(tv, m, 'si')} | {cell(tt, m, 'en')} | {cell(tt, m, 'si')} |" for m in list(PICKERS) + ["current_selection_whole_tables (before)"]) + f'''

The chat uses **{PICKER}** (best validation share of answers with all key facts). Settings: {SEL["n_passages"]} passage(s), up
to {SEL["max_units"]} units within {SEL["margin"]} of the best; age bonus {AGE_BONUS}; field-profile boost {PROFILE_BOOST} rank
positions. CPU cost: about {RERANK_MS:.0f} ms per question.

**Limits.** Labels only know numbers and variety codes, so questions answered in words are under-represented; the
"relevant" label can be noisy when a number appears in several rows.
'''
    (OUT / "model_card.md").write_text(card, encoding="utf-8")
print(json.dumps(config, indent=2))

{
  "n_passages": 1,
  "max_units": 3,
  "margin": 0.4,
  "picker": "fine_tuned_reranker",
  "lang_delta": 0.1,
  "age_bonus": 0.3,
  "profile_boost": 3,
  "max_length": 256,
  "base_model": "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1",
  "chosen_on": "validation split",
  "seed": 20261018
}
